# Kaggle ComfyUI GPU-as-a-Service

Self-contained notebook deploying ComfyUI as a persistent GPU service with automatic crash recovery and public access via ngrok.

**Do not hand-edit this notebook.** Regenerate via `scripts/generate_notebook.py`.

This notebook clones the project from GitHub, then runs installation and startup scripts.

In [ ]:
# Cell tag: clone — Pulls project files from GitHub
# Kaggle only uploads the notebook, so we must clone the repo
import subprocess
import os

REPO_URL = 'https://github.com/osumtr-web/kaggle-comfyui-queue.git'
WORKDIR = '/kaggle/working/kaggle-comfyui-queue'

if os.path.exists(WORKDIR):
    print(f'Repo already exists at {WORKDIR}, skipping clone')
else:
    print(f'Cloning {REPO_URL}...')
    result = subprocess.run(
        ['git', 'clone', REPO_URL, WORKDIR],
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        print(f'❌ Clone failed: {result.stderr}')
        raise SystemExit(1)
    print(f'✅ Cloned to {WORKDIR}')


In [ ]:
# Cell tag: install — Installs ComfyUI, dependencies, and curated models
# Wraps install.sh which is idempotent (skips if .kgpu_installed flag exists)
!bash /kaggle/working/kaggle-comfyui-queue/install.sh --non-interactive

In [ ]:
# Cell tag: model — Downloads additional models via model_registry.py
# Moonmaster Krea2 Turbo is already downloaded by install.sh;
# this cell demonstrates how to add more models post-install.
!python /kaggle/working/kaggle-comfyui-queue/scripts/model_registry.py install \
    --repo stabilityai/sdxl-turbo \
    --filename sd_xl_turbo_1.0_fp16.safetensors \
    --type diffusion_image \
    --tag sdxl,turbo,fallback

In [ ]:
# Cell tag: start — Launches all services (ComfyUI, watchdog, queue API, consumer, ngrok)
# Wraps scripts/start.sh which handles stale process cleanup, readiness probes,
# ngrok URL discovery, and auth credential generation.
!bash /kaggle/working/kaggle-comfyui-queue/start.sh

In [ ]:
# Cell tag: verify — Confirms deployment health and prints access URLs
import subprocess, json, time

# Wait for queue API health endpoint
for i in range(30):
    try:
        r = subprocess.run(
            ['curl', '-sf', 'http://localhost:8189/api/health'],
            capture_output=True, text=True, timeout=5
        )
        if r.returncode == 0:
            health = json.loads(r.stdout)
            print(f'✅ Queue API healthy: {json.dumps(health, indent=2)}')
            break
    except Exception:
        pass
    time.sleep(2)
else:
    print('⚠️ Queue API not responding after 60s — check logs')

# Print ngrok URL
try:
    with open('/kaggle/working/ngrok_url.txt') as f:
        url = f.read().strip()
    print(f'🌐 Public URL: {url}')
    print(f'🔑 Auth: kaggle:<password printed in startup logs>')
except FileNotFoundError:
    print('⚠️ ngrok_url.txt not found — ngrok may still be starting')

In [ ]:
# Cell tag: gpu-guard — Manually stop GPU session to preserve 30h/week quota
# The GPU Session Guard runs automatically via start.sh and auto-stops
# after 30 min of idle time. Use this cell to stop immediately.
import subprocess
result = subprocess.run(
    ['python3', '/kaggle/working/kaggle-comfyui-queue/scripts/gpu_session_guard.py',
     '--idle-timeout', '0'],
    capture_output=True, text=True, timeout=30
)
print(result.stdout)
if result.stderr:
    print(result.stderr)